# LACCI rerun notebook

This notebook reruns the experiments for the revised LACCI paper using `open_flow_minimal_CC18_shap_metrics.py`.

The notebook is organized around the revised paper design:

1. load CC18 tasks/evaluations and cache flows  
2. restrict the model family to the selected anchor/explainability models  
3. run the main task-group experiments for:
   - TF-IDF + metafeatures
   - MiniLM + metafeatures
   - optional metafeatures-only baseline
4. export predictive summaries and SHAP summaries
5. optionally run robustness / appendix experiments



In [ ]:

import os
import json
import joblib
import pandas as pd
from pathlib import Path

import openml
import open_flow_minimal_CC18_shap_metrics as wf

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


## Configuration

Edit only this cell if you want to change cache locations, CV setup, model subset, or which experiments to run.


In [ ]:

# Paths
CACHE_DIR = "minimal_cache_cc18"
RESULTS_DIR = "minimal_results_cc18_lacci"
FLOWS_CACHE = "flows_cc18.joblib"

# Main paper settings
CV_FOLDS = 5
RANDOM_STATE = 42
AGG_MODE = "mean"

# SHAP settings
COMPUTE_SHAP = True
SHAP_BACKGROUND_SIZE = 200
SHAP_EVAL_SIZE = 100
COMPUTE_SHAP_INTERACTIONS = False   # set True if you want the interaction section
SHAP_INTERACTION_EVAL_SIZE = 50
SHAP_TOP_K = 10
SHAP_SPARSITY_MASS = 0.8

# Main LACCI experiments
MAIN_EXPERIMENTS = [
    {"name": "tfidf_meta", "text_mode": "tfidf", "use_task_metafeatures": True},
    {"name": "minilm_meta", "text_mode": "minilm", "use_task_metafeatures": True},
]

# Small predictive baseline (optional but useful)
BASELINE_EXPERIMENTS = [
    {"name": "none_meta", "text_mode": "none", "use_task_metafeatures": True},
]

# Optional appendix / sanity-check experiments
RUN_BASELINE = True
RUN_ROW_KFOLD_APPENDIX = False

# Restrict models to the revised paper
SELECTED_MODELS = {"hist_gbrt", "random_forest"}


## Monkeypatch model selection

`run_cc18_pipeline()` uses `get_models()` internally.  
This cell restricts it to the models used in the revised paper.


In [ ]:

_original_get_models = wf.get_models

def _selected_get_models(random_state: int = 42):
    models = _original_get_models(random_state=random_state)
    return {k: v for k, v in models.items() if k in SELECTED_MODELS}

wf.get_models = _selected_get_models

wf.get_models(RANDOM_STATE)


## Load CC18 tasks and evaluations

In [ ]:

cache = wf.DiskCache(CACHE_DIR)

bundle = wf.load_cc18_from_openml(
    function="predictive_accuracy",
    suite_name="OpenML-CC18",
    cache=cache,
)

tasks_df = bundle["tasks_df"]
evals_df = bundle["evals_df"]

tasks_df.shape, evals_df.shape


## Load or cache flows

The flow list is derived from the evaluation table, so there is no dependency on a previously defined `used_flow_ids`.


In [ ]:

used_flow_ids = sorted(pd.Series(evals_df["flow_id"].unique()).dropna().astype(int).tolist())

if Path(FLOWS_CACHE).exists():
    flows = joblib.load(FLOWS_CACHE)
else:
    flows = openml.flows.list_flows(
        flow=used_flow_ids,
        output_format="dict",
    )
    joblib.dump(flows, FLOWS_CACHE)

len(flows), len(used_flow_ids)


## Main task-group experiments for the revised paper

These are the main reruns to support the LACCI version:
- TF-IDF + metafeatures
- MiniLM + metafeatures
- optional metafeatures-only baseline

All are run with `task_group_kfold` as the main protocol.


In [ ]:

experiment_outputs = {}

configs = list(MAIN_EXPERIMENTS)
if RUN_BASELINE:
    configs += BASELINE_EXPERIMENTS

for cfg in configs:
    print(f"Running {cfg['name']} ...")
    out = wf.run_cc18_pipeline(
        flows=flows,
        tasks_df=tasks_df,
        evals_df=evals_df,
        agg_mode=AGG_MODE,
        text_mode=cfg["text_mode"],
        use_task_metafeatures=cfg["use_task_metafeatures"],
        run_row_kfold=False,
        run_task_group_kfold=True,
        cv_folds=CV_FOLDS,
        random_state=RANDOM_STATE,
        compute_shap=COMPUTE_SHAP,
        shap_background_size=SHAP_BACKGROUND_SIZE,
        shap_eval_size=SHAP_EVAL_SIZE,
        compute_shap_interactions=COMPUTE_SHAP_INTERACTIONS,
        shap_interaction_eval_size=SHAP_INTERACTION_EVAL_SIZE,
        shap_top_k=SHAP_TOP_K,
        shap_sparsity_mass=SHAP_SPARSITY_MASS,
        cache_dir=CACHE_DIR,
        results_dir=RESULTS_DIR,
    )
    experiment_outputs[cfg["name"]] = out

list(experiment_outputs.keys())


## Predictive anchor table

This cell builds the compact predictive anchor table for the paper.


In [ ]:

anchor_rows = []
for exp_name, out in experiment_outputs.items():
    s = out["summary"].copy()
    s = s[s["split_mode"] == "task_group_kfold"].copy()
    s["experiment"] = exp_name
    anchor_rows.append(s)

anchor_df = pd.concat(anchor_rows, ignore_index=True)
anchor_df = anchor_df[
    ["experiment", "model", "split_mode", "r2_mean", "r2_std", "mae_mean", "mae_std", "mse_mean", "mse_std"]
].sort_values(["experiment", "r2_mean"], ascending=[True, False])

anchor_df


In [ ]:

anchor_path = Path(RESULTS_DIR) / "lacci_predictive_anchor.csv"
anchor_df.to_csv(anchor_path, index=False)
anchor_path


## SHAP summary table

This cell collects the per-run `shap_summary_metrics.csv` files into one paper-level table.


In [ ]:

shap_rows = []

for exp_name, out in experiment_outputs.items():
    shap_path = Path(out["run_dir"]) / "shap_summary_metrics.csv"
    if shap_path.exists():
        df = pd.read_csv(shap_path)
        df["experiment"] = exp_name
        shap_rows.append(df)

if shap_rows:
    shap_summary_all = pd.concat(shap_rows, ignore_index=True)
    shap_summary_all = shap_summary_all.sort_values(["experiment", "split_mode", "model"])
else:
    shap_summary_all = pd.DataFrame()

shap_summary_all


In [ ]:

shap_summary_path = Path(RESULTS_DIR) / "lacci_shap_summary_metrics.csv"
if len(shap_summary_all) > 0:
    shap_summary_all.to_csv(shap_summary_path, index=False)
shap_summary_path


## Global SHAP top-k tables

Useful for reporting:
- top global explanatory features per experiment/model
- stability of global top-k features across folds


In [ ]:

global_topk_tables = {}
global_stability_tables = {}

for exp_name, out in experiment_outputs.items():
    run_dir = Path(out["run_dir"])
    for model_name in SELECTED_MODELS:
        topk_path = run_dir / f"global_shap_topk_task_group_kfold_{model_name}.csv"
        stab_path = run_dir / f"global_shap_stability_task_group_kfold_{model_name}.csv"

        if topk_path.exists():
            global_topk_tables[(exp_name, model_name)] = pd.read_csv(topk_path)
        if stab_path.exists():
            global_stability_tables[(exp_name, model_name)] = pd.read_csv(stab_path)

list(global_topk_tables.keys()), list(global_stability_tables.keys())


In [ ]:

# Example: inspect one top-k table
example_key = next(iter(global_topk_tables.keys())) if global_topk_tables else None
if example_key is not None:
    print("Example:", example_key)
    display(global_topk_tables[example_key].head(20))


## Local SHAP metrics tables

These are the main explainability metrics for the revised paper:
- readability
- sparsity


In [ ]:

local_metric_tables = {}

for exp_name, out in experiment_outputs.items():
    run_dir = Path(out["run_dir"])
    for model_name in SELECTED_MODELS:
        local_path = run_dir / f"local_shap_metrics_task_group_kfold_{model_name}.csv"
        if local_path.exists():
            local_metric_tables[(exp_name, model_name)] = pd.read_csv(local_path)

list(local_metric_tables.keys())


In [ ]:

local_summary_rows = []
for (exp_name, model_name), df in local_metric_tables.items():
    local_summary_rows.append({
        "experiment": exp_name,
        "model": model_name,
        "n": len(df),
        "readability_mean": df["readability"].mean(),
        "readability_std": df["readability"].std(ddof=0),
        "sparsity_mean": df["sparsity"].mean(),
        "sparsity_std": df["sparsity"].std(ddof=0),
    })

local_summary_df = pd.DataFrame(local_summary_rows).sort_values(["experiment", "model"])
local_summary_df


In [ ]:

local_summary_path = Path(RESULTS_DIR) / "lacci_local_shap_metrics_summary.csv"
local_summary_df.to_csv(local_summary_path, index=False)
local_summary_path


## Optional interaction metrics

Only populated if `COMPUTE_SHAP_INTERACTIONS=True`.


In [ ]:

interaction_tables = {}

for exp_name, out in experiment_outputs.items():
    run_dir = Path(out["run_dir"])
    for model_name in SELECTED_MODELS:
        interaction_path = run_dir / f"interaction_metrics_task_group_kfold_{model_name}.csv"
        if interaction_path.exists():
            interaction_tables[(exp_name, model_name)] = pd.read_csv(interaction_path)

if interaction_tables:
    interaction_summary = pd.concat(
        [df.assign(experiment=exp, model=model) for (exp, model), df in interaction_tables.items()],
        ignore_index=True
    )
    interaction_summary
else:
    interaction_summary = pd.DataFrame()
    interaction_summary


## Optional appendix robustness runs

Turn `RUN_ROW_KFOLD_APPENDIX=True` in the configuration cell if you want the row-wise split sanity check.


In [ ]:

appendix_outputs = {}

if RUN_ROW_KFOLD_APPENDIX:
    appendix_configs = list(MAIN_EXPERIMENTS)
    for cfg in appendix_configs:
        print(f"Running appendix row_kfold for {cfg['name']} ...")
        out = wf.run_cc18_pipeline(
            flows=flows,
            tasks_df=tasks_df,
            evals_df=evals_df,
            agg_mode=AGG_MODE,
            text_mode=cfg["text_mode"],
            use_task_metafeatures=cfg["use_task_metafeatures"],
            run_row_kfold=True,
            run_task_group_kfold=False,
            cv_folds=CV_FOLDS,
            random_state=RANDOM_STATE,
            compute_shap=False,
            cache_dir=CACHE_DIR,
            results_dir=RESULTS_DIR,
        )
        appendix_outputs[cfg["name"]] = out

list(appendix_outputs.keys())


## Final export overview

In [ ]:

export_paths = {
    "predictive_anchor": str(anchor_path),
    "shap_summary": str(shap_summary_path),
    "local_summary": str(local_summary_path),
}

if len(interaction_summary) > 0:
    interaction_summary_path = Path(RESULTS_DIR) / "lacci_interaction_summary.csv"
    interaction_summary.to_csv(interaction_summary_path, index=False)
    export_paths["interaction_summary"] = str(interaction_summary_path)

pd.DataFrame([export_paths]).T.rename(columns={0: "path"})
